# PYNQ-Z2 HLS m_axi Fake Capture Demo

This notebook loads `base_add.bit`, allocates a DDR buffer, lets the HLS IP write fake two-channel samples through `m_axi_GMEM`, then reads and plots the buffer from PS Python.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

MAX_SAMPLE_N = 1024
BUFFER_WORDS = MAX_SAMPLE_N * 2
SAMPLE_COUNT = 1024

In [ ]:
overlay = Overlay('base_add.bit')
ip = overlay.base_add_0

print(overlay.ip_dict.keys())
ip.register_map

In [ ]:
buf = allocate(shape=(BUFFER_WORDS,), dtype=np.int32)
buf[:] = 0
buf.flush()

sample_count = min(SAMPLE_COUNT, MAX_SAMPLE_N)
phys_addr = int(buf.physical_address)

print(f'Buffer physical address: 0x{phys_addr:08x}')
print(f'Sample count: {sample_count}')

In [ ]:
# HLS report / register_map:
#   0x00 = AP_CTRL
#   0x10 = buffer_r physical address
#   0x18 = sample_count
#
# 注意：register_map 里名字是 buffer_r，不是 buffer。
# 为了避免名字不匹配，这里直接按地址写寄存器。

AP_CTRL_ADDR = 0x00
BUFFER_ADDR = 0x10
SAMPLE_COUNT_ADDR = 0x18

AP_START = 0x01
AP_DONE = 0x02

# 先把 buffer 填成明显的哨兵值，方便判断 HLS 是否真的写入 DDR。
buf[:] = -12345
buf.flush()

ip.write(BUFFER_ADDR, phys_addr & 0xFFFFFFFF)
ip.write(SAMPLE_COUNT_ADDR, sample_count)

print("buffer physical address written:", hex(phys_addr & 0xFFFFFFFF))
print("sample_count written:", sample_count)
print("CTRL before:", hex(ip.read(AP_CTRL_ADDR)))

ip.write(AP_CTRL_ADDR, AP_START)

start = time.time()
while (ip.read(AP_CTRL_ADDR) & AP_DONE) == 0:
    if time.time() - start > 2.0:
        raise TimeoutError("HLS IP timeout waiting for ap_done")

print("CTRL after:", hex(ip.read(AP_CTRL_ADDR)))

buf.invalidate()
print("Capture done")


In [ ]:
ch0 = np.array(buf[:sample_count], dtype=np.int32)
ch1 = np.array(buf[MAX_SAMPLE_N:MAX_SAMPLE_N + sample_count], dtype=np.int32)

print('CH0 first 8:', ch0[:8])
print('CH1 first 8:', ch1[:8])

assert np.all(ch0 == np.arange(sample_count, dtype=np.int32))
assert np.all(ch1 == np.arange(sample_count - 1, -1, -1, dtype=np.int32))
print('FINAL: PASS')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ch0, label='CH0 fake ramp')
plt.plot(ch1, label='CH1 fake reverse ramp')
plt.title('PYNQ-Z2 HLS m_axi Fake Capture')
plt.xlabel('Sample Index')
plt.ylabel('Sample Value')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
buf.freebuffer()